# 🏠 Ames Housing Dataset — معرفی و آشنایی با داده‌ها

**درس:** دیداری‌سازی داده‌ها  
**دانشجو:** سارا دهقانی  
**دیتاست:** Ames Housing Dataset

---

## 📌 درباره دیتاست

این دیتاست توسط **Dean De Cock** در سال ۲۰۱۱ منتشر شده و شامل اطلاعات فروش واقعی خانه‌های شهر **Ames، Iowa** آمریکا بین سال‌های ۲۰۰۶ تا ۲۰۱۰ است.  
هدف اصلی: **پیش‌بینی قیمت فروش خانه (SalePrice)** بر اساس ۸۰ ویژگی مختلف.

## 1️⃣ بارگذاری کتابخانه‌ها و داده‌ها

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# تنظیمات نمایش فارسی و ظاهری
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'
sns.set_style('whitegrid')
BLUE   = '#378ADD'
GREEN  = '#1D9E75'
ORANGE = '#D85A30'
PURPLE = '#7F77DD'
AMBER  = '#BA7517'
PALETTE = [BLUE, GREEN, ORANGE, PURPLE, AMBER]

# بارگذاری دیتاست
df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
print(f'✅ دیتاست بارگذاری شد: {df.shape[0]} خانه، {df.shape[1]} ستون')

## 2️⃣ نگاه اول به داده‌ها

In [ ]:
# نمایش ۵ ردیف اول
df.head()

In [ ]:
# ابعاد و نوع داده‌ها
print('=' * 50)
print(f'تعداد نمونه‌ها (خانه):  {df.shape[0]}')
print(f'تعداد ویژگی‌ها:          {df.shape[1] - 1}  (+ 1 ستون هدف)')
print('=' * 50)
num_cols = df.select_dtypes(include='number').columns.drop(['Id','SalePrice']).tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'ویژگی‌های عددی:   {len(num_cols)}')
print(f'ویژگی‌های متنی:    {len(cat_cols)}')
print(f'ستون هدف:         SalePrice')

In [ ]:
# اطلاعات کلی
df.info()

## 3️⃣ آمار توصیفی ستون هدف (SalePrice)

In [ ]:
sp = df['SalePrice']
print('آمار توصیفی SalePrice')
print('=' * 40)
print(f'کمترین قیمت:      ${sp.min():>10,.0f}')
print(f'چارک اول (Q1):    ${sp.quantile(0.25):>10,.0f}')
print(f'میانه:            ${sp.median():>10,.0f}')
print(f'میانگین:          ${sp.mean():>10,.0f}')
print(f'چارک سوم (Q3):    ${sp.quantile(0.75):>10,.0f}')
print(f'بیشترین قیمت:     ${sp.max():>10,.0f}')
print(f'انحراف معیار:     ${sp.std():>10,.0f}')
print(f'چولگی (Skewness): {sp.skew():>10.2f}  ← توزیع راست‌کج')

## 4️⃣ توزیع قیمت فروش (هیستوگرام)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- هیستوگرام قیمت اصلی ---
ax = axes[0]
ax.hist(df['SalePrice'], bins=40, color=BLUE, edgecolor='white', linewidth=0.5)
ax.axvline(df['SalePrice'].mean(),   color='#185FA5', linewidth=2, linestyle='--', label=f"میانگین: ${df['SalePrice'].mean():,.0f}")
ax.axvline(df['SalePrice'].median(), color=GREEN,     linewidth=2, linestyle='-',  label=f"میانه:   ${df['SalePrice'].median():,.0f}")
ax.set_title('توزیع قیمت فروش (SalePrice)', fontsize=13, pad=12)
ax.set_xlabel('قیمت (دلار)', fontsize=11)
ax.set_ylabel('تعداد خانه', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend(fontsize=10)

# --- توزیع لگاریتمی ---
ax2 = axes[1]
log_price = np.log1p(df['SalePrice'])
ax2.hist(log_price, bins=40, color=GREEN, edgecolor='white', linewidth=0.5)
ax2.axvline(log_price.mean(),   color='#0F6E56', linewidth=2, linestyle='--', label=f'میانگین: {log_price.mean():.2f}')
ax2.axvline(log_price.median(), color=BLUE,      linewidth=2, linestyle='-',  label=f'میانه:   {log_price.median():.2f}')
ax2.set_title('توزیع لگاریتم قیمت — log(SalePrice+1)', fontsize=13, pad=12)
ax2.set_xlabel('log(SalePrice + 1)', fontsize=11)
ax2.set_ylabel('تعداد خانه', fontsize=11)
ax2.legend(fontsize=10)

plt.suptitle('چولگی توزیع قیمت و اصلاح آن با تبدیل لگاریتمی', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print(f'چولگی اصلی: {df["SalePrice"].skew():.2f}  →  بعد از log: {log_price.skew():.2f}')

## 5️⃣ بررسی مقادیر گمشده (Missing Values)

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)

missing_df = pd.DataFrame({'تعداد': missing, 'درصد': missing_pct})
print(f'تعداد ستون‌های دارای مقدار گمشده: {len(missing_df)}')
print()
print(missing_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = [ORANGE if p > 50 else AMBER if p > 20 else GREEN for p in missing_pct.values]
bars = ax.barh(missing_df.index, missing_df['درصد'], color=colors, edgecolor='white', linewidth=0.5)

for bar, pct in zip(bars, missing_df['درصد']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct}%', va='center', fontsize=9)

ax.axvline(50, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='50%')
ax.set_xlabel('درصد مقادیر گمشده', fontsize=11)
ax.set_title('مقادیر گمشده به تفکیک ستون', fontsize=13, pad=12)
ax.set_xlim(0, 105)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=ORANGE, label='بیش از 50% (احتمالاً «ندارم»)'),
    Patch(facecolor=AMBER,  label='20 تا 50%'),
    Patch(facecolor=GREEN,  label='کمتر از 20% (قابل جبران)')
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()

## 6️⃣ همبستگی ویژگی‌های عددی با SalePrice

In [ ]:
corr = df[num_cols + ['SalePrice']].corr()['SalePrice'].drop('SalePrice')
corr_sorted = corr.abs().sort_values(ascending=False).head(14)
corr_vals   = corr[corr_sorted.index]

fig, ax = plt.subplots(figsize=(10, 6))
colors = [BLUE if v > 0.6 else '#85B7EB' if v > 0.4 else '#B5D4F4' for v in corr_vals.values]
ax.barh(corr_sorted.index[::-1], corr_vals.values[::-1], color=colors[::-1], edgecolor='white', linewidth=0.5)

for i, (idx, val) in enumerate(zip(corr_sorted.index[::-1], corr_vals.values[::-1])):
    ax.text(val + 0.01, i, f'{val:.2f}', va='center', fontsize=9)

ax.set_xlim(0, 1)
ax.set_xlabel('ضریب همبستگی پیرسون', fontsize=11)
ax.set_title('قوی‌ترین همبستگی‌ها با SalePrice', fontsize=13, pad=12)
ax.axvline(0.6, color='gray', linestyle='--', linewidth=1, alpha=0.5)
plt.tight_layout()
plt.show()

## 7️⃣ توزیع نوع خانه و سبک معماری

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- نوع خانه ---
bldg_counts = df['BldgType'].value_counts()
axes[0].pie(bldg_counts.values, labels=bldg_counts.index, autopct='%1.1f%%',
            colors=PALETTE, startangle=140, textprops={'fontsize': 9})
axes[0].set_title('نوع خانه (BldgType)', fontsize=12)

# --- سبک معماری ---
style_counts = df['HouseStyle'].value_counts()
axes[1].barh(style_counts.index[::-1], style_counts.values[::-1],
             color=BLUE, edgecolor='white', linewidth=0.5)
for i, v in enumerate(style_counts.values[::-1]):
    axes[1].text(v + 5, i, str(v), va='center', fontsize=9)
axes[1].set_title('سبک معماری (HouseStyle)', fontsize=12)
axes[1].set_xlabel('تعداد')

# --- زون‌بندی ---
zone_counts = df['MSZoning'].value_counts()
axes[2].bar(zone_counts.index, zone_counts.values,
            color=PALETTE, edgecolor='white', linewidth=0.5)
for i, v in enumerate(zone_counts.values):
    axes[2].text(i, v + 10, str(v), ha='center', fontsize=9)
axes[2].set_title('زون‌بندی (MSZoning)', fontsize=12)
axes[2].set_ylabel('تعداد')

plt.suptitle('ترکیب کلی خانه‌های دیتاست', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8️⃣ توزیع زمانی — سال ساخت و فروش

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# سال ساخت
axes[0].hist(df['YearBuilt'], bins=30, color=PURPLE, edgecolor='white', linewidth=0.5)
axes[0].set_title('توزیع سال ساخت خانه‌ها', fontsize=12)
axes[0].set_xlabel('سال ساخت', fontsize=11)
axes[0].set_ylabel('تعداد خانه', fontsize=11)

# میانگین قیمت به تفکیک سال فروش
yr_price = df.groupby('YrSold')['SalePrice'].mean()
axes[1].bar(yr_price.index.astype(str), yr_price.values / 1000,
            color=GREEN, edgecolor='white', linewidth=0.5)
for i, v in enumerate(yr_price.values):
    axes[1].text(i, v/1000 + 0.5, f'${v/1000:.0f}K', ha='center', fontsize=9)
axes[1].set_title('میانگین قیمت به تفکیک سال فروش', fontsize=12)
axes[1].set_xlabel('سال فروش', fontsize=11)
axes[1].set_ylabel('میانگین قیمت (هزار دلار)', fontsize=11)

plt.tight_layout()
plt.show()

## 9️⃣ محله‌های پرتکرار و قیمت میانه آن‌ها

In [ ]:
neigh_price = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors_n = [BLUE if v > 200000 else GREEN if v > 150000 else AMBER for v in neigh_price.values]
ax.barh(neigh_price.index[::-1], neigh_price.values[::-1] / 1000, color=colors_n[::-1],
        edgecolor='white', linewidth=0.5)

for i, v in enumerate(neigh_price.values[::-1]):
    ax.text(v/1000 + 1, i, f'${v/1000:.0f}K', va='center', fontsize=8)

ax.axvline(df['SalePrice'].median()/1000, color='gray', linestyle='--',
           linewidth=1.5, label=f'میانه کل: ${df["SalePrice"].median()/1000:.0f}K')
ax.set_xlabel('میانه قیمت فروش (هزار دلار)', fontsize=11)
ax.set_title('میانه قیمت فروش به تفکیک محله', fontsize=13, pad=12)
ax.legend(fontsize=10)
ax.set_xlim(0, 360)
plt.tight_layout()
plt.show()

## 🔟 خلاصه نتایج معرفی دیتاست

| موضوع | یافته |
|---|---|
| اندازه دیتاست | 1460 خانه، 80 ویژگی ورودی |
| ستون هدف | SalePrice (بین 34,900 تا 755,000 دلار) |
| توزیع قیمت | راست‌کج (skew=1.88) — نیاز به تبدیل لگاریتمی |
| مهم‌ترین ویژگی | OverallQual (همبستگی 0.79) |
| مقادیر گمشده | 19 ستون — اکثراً به معنای «ندارم» |
| گران‌ترین محله | NoRidge و NridgHt |
| رایج‌ترین نوع خانه | تک‌خانواده یک‌طبقه (1Story, 1Fam) |